# Emotion probes on SWE-bench coding trajectories — Colab runner

End-to-end runner for the `experiment/swebench_coding_emotions` pipeline. Each phase is its own cell so a Colab session timeout doesn't cost you the run — outputs are persisted to Google Drive, and every phase checkpoints to disk.

**Recommended runtime:** Colab Pro+ A100 40 GB (or larger). The build-probes phase on DeepSWE-Preview is ~30 h on a single H100 at 4-bit; expect proportionally longer on smaller GPUs. Replay is much shorter (24 trajectories).

**Order:** run cells top-to-bottom on a fresh runtime, or skip ahead — every phase is idempotent and resumable from Drive.

Phases:
1. Setup (GPU check, clone repo, install, HF login, mount Drive)
2. Build emotion probes on DeepSWE-Preview
3. Augment neutrals + refit PCA denoising
4. Fetch SWE-bench trajectories
5. Replay trajectories through the model
6. Analyze (statistical tests)
7. Visualize
8. Inspect a few figures inline

## 1. Setup

In [ ]:
# 1a. Verify the runtime has a GPU and report VRAM.
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 1b. Mount Google Drive so output persists across sessions.
# A session timeout will lose /content but keep /content/drive/MyDrive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1c. Configure paths. Edit DRIVE_DIR if you want a different output location.
import os
from pathlib import Path

REPO_URL    = 'https://github.com/V-Coding/emotion-probes.git'
BRANCH      = 'claude/emotion-probes-on-coding-tasks'
REPO_DIR    = Path('/content/emotion-probes')
EXP_DIR     = REPO_DIR / 'experiment' / 'swebench_coding_emotions'
DRIVE_DIR   = Path('/content/drive/MyDrive/emotion-probes-experiment')
DRIVE_OUT   = DRIVE_DIR / 'output'              # all artefacts land here
DRIVE_HFCACHE = DRIVE_DIR / 'hf_cache'          # model weights cache (saves redownload)

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
DRIVE_HFCACHE.mkdir(parents=True, exist_ok=True)

# Point HuggingFace at Drive so we don't redownload the 32B weights every session.
os.environ['HF_HOME'] = str(DRIVE_HFCACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(DRIVE_HFCACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(DRIVE_HFCACHE / 'hub')

print('Repo:    ', REPO_DIR)
print('Output:  ', DRIVE_OUT)
print('HF cache:', DRIVE_HFCACHE)

In [ ]:
# 1d. Clone repo and check out the experiment branch (skip if already present).
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only

# Symlink the experiment's output/ to Drive so checkpointing survives session restarts.
exp_output = EXP_DIR / 'output'
if exp_output.is_symlink() or exp_output.exists():
    if not exp_output.is_symlink():
        # If it's a real dir from a prior local run, move contents into Drive once.
        import shutil
        for child in exp_output.iterdir():
            dest = DRIVE_OUT / child.name
            if not dest.exists():
                shutil.move(str(child), str(dest))
        exp_output.rmdir()
        exp_output.symlink_to(DRIVE_OUT)
else:
    exp_output.symlink_to(DRIVE_OUT)

!ls -la {EXP_DIR}/output

In [ ]:
# 1e. Install the root emotion-probes package (with 4-bit quantization extras) and the experiment package.
# Using --quiet to keep output readable; remove if you need to debug a build failure.
%pip install --quiet -e '{REPO_DIR}[quantize]'
%pip install --quiet -e '{EXP_DIR}'

# Quick sanity check — both CLIs should be on PATH.
!emotion-probes --help | head -20
print('---')
!swebench-emotions --help | head -20

In [ ]:
# 1f. HuggingFace login. DeepSWE-Preview is public, but logging in raises rate limits
# and is required if you ever swap the config to a gated model (Gemma, Llama, etc.).
# Get a token at https://huggingface.co/settings/tokens — read access is enough.
from huggingface_hub import login
login()  # paste token when prompted; it is stored under HF_HOME on Drive

In [ ]:
# 1g. cd into the experiment dir — config files use relative paths and the scripts
# expect this CWD. From here on every shell cell runs from EXP_DIR.
%cd {EXP_DIR}
!cat config/deepswe.yaml

## 2. Build probes — `01_build_probes.sh` equivalent

This is the longest phase: generate emotion-labeled stories, generate neutral dialogues, extract activations at layer 42, denoise via PCA, then run analysis + visualization for the *raw* probes.

**Each subcommand checkpoints**, so a Colab session crash is recoverable — just re-run the cell. The outputs land under `output/probes/`.

In [ ]:
CFG = 'config/deepswe.yaml'

!emotion-probes generate-stories    --config {CFG}
!emotion-probes generate-neutral    --config {CFG}
!emotion-probes extract-activations --config {CFG}
!emotion-probes compute-vectors     --config {CFG}
!emotion-probes analyze             --config {CFG}
!emotion-probes visualize           --config {CFG}

!ls -la output/probes/

## 3. Augment neutrals + refit PCA — `01b_augment_neutrals.sh`

Generates trajectory-shaped neutral activations (chat templates, code blocks, diffs, `<think>` blocks) and refits the PCA denoising basis against them. Produces `emotion_vectors_augmented_layer_42.safetensors`, which the replay phase will use when `--variant augmented` is set.

In [ ]:
!swebench-emotions augment --config {CFG}

## 4. Fetch trajectories — `02_fetch_trajectories.sh`

Downloads DeepSWE's public SWE-bench Verified trajectories from the `swe-bench-submissions` S3 bucket and stratified-samples 24 tasks (6 each across {pass, fail} × {easy, hard}). Cached under `output/swebench/raw/`.

Set `N_TASKS` higher (e.g. 48) if you want a larger sample — but replay time scales linearly.

In [ ]:
N_TASKS = 24
!swebench-emotions fetch --config {CFG} --n-tasks {N_TASKS}

## 5. Replay — `03_replay.sh`

Teacher-forced replay of each trajectory through DeepSWE-Preview. Captures per-token residual activations at layer 42 and projects them onto the emotion directions. `--variant augmented` uses the PCA basis refit in phase 3.

Resumable: per-task scores are written one `.safetensors` at a time, so an interrupted run picks up where it left off when re-run.

In [ ]:
VARIANT = 'augmented'  # or 'denoised' or 'raw'
!swebench-emotions replay --config {CFG} --variant {VARIANT}

## 6. Analyze — `04_analyze.sh`

Aggregates per-token probe scores into per-task summaries (`mean`, `max`, `p90`), drops the first 50 tokens and special-token positions, then runs Mann-Whitney U and OLS group-comparison tests across {pass vs fail} × {easy vs hard}. Outputs land in `output/swebench/analysis/{variant}/`.

In [ ]:
!swebench-emotions analyze --config {CFG} --variant {VARIANT}
!ls output/swebench/analysis/{VARIANT}/

## 7. Visualize — `05_visualize.sh`

Generates timelines, box plots, decile curves, and a heatmap from the per-task scores. Outputs in `output/swebench/figures/{variant}/`.

In [ ]:
!swebench-emotions viz --config {CFG} --variant {VARIANT}
!ls output/swebench/figures/{VARIANT}/

## 8. Inspect results

In [ ]:
# Show summary stats and the first few figures inline.
import json
from pathlib import Path
from IPython.display import Image, display, Markdown

summary_path = Path(f'output/swebench/analysis/{VARIANT}/summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(Markdown('### Summary'))
    print(json.dumps(summary, indent=2)[:4000])

fig_dir = Path(f'output/swebench/figures/{VARIANT}')
for png in sorted(fig_dir.glob('*.png'))[:6]:
    display(Markdown(f'### {png.name}'))
    display(Image(str(png)))

In [ ]:
# Optional — zip the experiment outputs for download.
# The full output/ also lives on Drive at DRIVE_OUT, so this is just for convenience.
import shutil
archive = shutil.make_archive('/content/swebench_emotions_output', 'zip', root_dir='output')
print('Wrote', archive)
from google.colab import files
files.download(archive)